# Notebook 4: Agregación con Apache Spark (comuna × año)

Toma el dataset evento-a-evento cruzado con clima (`03_cruzar_incendios_era5.ipynb`,
60.530 eventos con `t2m_c`, `tp_mm`, anomalías mensuales) y lo procesa con **PySpark** para
producir una tabla agregada comuna-año: la unidad de análisis que le sirve tanto a
CONAF/SENAPRED como a una aseguradora (no el evento individual, no el país completo).

Esto cubre el requisito no negociable del curso: **procesamiento con Apache Spark sobre un
dataset real**. El dataset de entrada es pequeño (60k filas) para justificar un clúster
distribuido — se usa Spark en modo local (`local[*]`) para demostrar el patrón de
ETL/agregación (DataFrame API, `groupBy`, funciones de ventana) que escalaría igual si el
dataset creciera a millones de filas (ej. incorporando el resto de temporadas CONAF o datos
satelitales de mayor resolución).

**Requiere como entrada** el output de `03_cruzar_incendios_era5.ipynb`
(`incendios_conaf_era5_2010_2020.parquet`), ya en tu Google Drive.


## 1. Montar Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Instalar PySpark

In [2]:
!pip install -q pyspark
print('✓ PySpark instalado')


✓ PySpark instalado


## 3. Configuración — EDITA la ruta antes de correr

In [3]:
import os

# ⚠️ Debe ser la MISMA carpeta base usada en el notebook 03
DRIVE_BASE = '/content/drive/MyDrive/BigDataSIC'

# Se lee el CSV, no el parquet: pandas/pyarrow guarda fecha_evento como timestamp de
# nanosegundos, y el lector de Parquet de Spark no soporta esa precisión
# (AnalysisException: PARQUET_TYPE_ILLEGAL ... TIMESTAMP(NANOS)). El CSV no lleva esa
# metadata de tipo, así que evita el choque por completo.
INPUT_CSV  = f'{DRIVE_BASE}/datos_procesados/incendios_conaf_era5_2010_2020.csv'
OUTPUT_DIR = f'{DRIVE_BASE}/datos_procesados/spark_comuna_anio'
os.makedirs(f'{DRIVE_BASE}/datos_procesados', exist_ok=True)

print('Entrada :', INPUT_CSV)
print('Salida  :', OUTPUT_DIR)


Entrada : /content/drive/MyDrive/BigDataSIC/datos_procesados/incendios_conaf_era5_2010_2020.csv
Salida  : /content/drive/MyDrive/BigDataSIC/datos_procesados/spark_comuna_anio


## 4. Iniciar sesión de Spark

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('incendios_comuna_anio')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')  # dataset chico, no hacen falta 200 particiones por defecto
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('✓ Spark', spark.version, 'corriendo en', spark.sparkContext.master)


✓ Spark 4.0.4 corriendo en local[*]


## 5. Cargar el dataset cruzado (evento-a-evento)

In [5]:
df = spark.read.csv(INPUT_CSV, header=True, inferSchema=True)

print(f'✓ {df.count()} eventos cargados · {len(df.columns)} columnas')
df.printSchema()
df.select('region', 'comuna', 'año', 'mes', 'superficie_ha', 't2m_c', 'tp_mm',
          't2m_anomaly_mensual', 'tp_anomaly_mensual').show(5, truncate=False)


✓ 60530 eventos cargados · 34 columnas
root
 |-- region: string (nullable = true)
 |-- Provincia: string (nullable = true)
 |-- comuna: string (nullable = true)
 |-- Temporada: string (nullable = true)
 |-- Nombre: string (nullable = true)
 |-- fecha_evento: date (nullable = true)
 |-- hora_evento: timestamp (nullable = true)
 |-- Duración (minutos): double (nullable = true)
 |-- Alerta: string (nullable = true)
 |-- Escenario: string (nullable = true)
 |-- causa: string (nullable = true)
 |-- Superficie quemada: Pino A [ha]: double (nullable = true)
 |-- Superficie quemada: Pino B [ha]: double (nullable = true)
 |-- Superficie quemada: Pino C [ha]: double (nullable = true)
 |-- Superficie quemada: Eucalípto [ha]: double (nullable = true)
 |-- Superficie quemada: Otras plantas [ha]: double (nullable = true)
 |-- Superficie quemada: Arbolado [ha]: double (nullable = true)
 |-- Superficie quemada: Matorral [ha]: double (nullable = true)
 |-- Superficie quemada: Pastizal [ha]: double (nul

## 6. Limpieza y features

- Descarta filas sin `comuna`/`región` (no se pueden agregar a nivel comuna).
- Bucket de severidad por evento (umbrales propios del proyecto, no una clasificación
  oficial de CONAF — sirven para el EDA/dashboard, no para el modelo).
- Bandera de temporada de incendios (dic-mar).


In [6]:
df_clean = (
    df
    .filter(F.col('comuna').isNotNull() & F.col('region').isNotNull() & F.col('año').isNotNull())
    .withColumn('superficie_ha', F.col('superficie_ha').cast('double'))
    .withColumn('t2m_c', F.col('t2m_c').cast('double'))
    .withColumn('tp_mm', F.col('tp_mm').cast('double'))
    .withColumn('t2m_anomaly_mensual', F.col('t2m_anomaly_mensual').cast('double'))
    .withColumn('tp_anomaly_mensual', F.col('tp_anomaly_mensual').cast('double'))
    .withColumn('es_temporada_incendios', F.col('mes').isin([12, 1, 2, 3]).cast('int'))
    .withColumn(
        'severidad_evento',
        F.when(F.col('superficie_ha') < 1, 'menor')
         .when(F.col('superficie_ha') < 10, 'moderado')
         .when(F.col('superficie_ha') < 200, 'grave')
         .otherwise('catastrofico')
    )
)

descartadas = df.count() - df_clean.count()
print(f'Descartadas {descartadas} filas sin comuna/región/año')
print(f'✓ {df_clean.count()} eventos listos para agregar')
df_clean.groupBy('severidad_evento').count().orderBy(F.desc('count')).show()


Descartadas 0 filas sin comuna/región/año
✓ 60530 eventos listos para agregar
+----------------+-----+
|severidad_evento|count|
+----------------+-----+
|           menor|39966|
|        moderado|15983|
|           grave| 3994|
|    catastrofico|  587|
+----------------+-----+



## 7. Agregación comuna × año

Una fila = una comuna en un año: cuántos incendios, cuánta superficie quemada, y el
clima promedio/anómalo de esos eventos — la tabla base para el dashboard y para
detectar comunas con severidad o estrés hídrico fuera de lo normal.


In [7]:
agg_comuna_anio = (
    df_clean.groupBy('region', 'comuna', 'año')
    .agg(
        F.count('*').alias('n_incendios'),
        F.sum('superficie_ha').alias('superficie_total_ha'),
        F.avg('superficie_ha').alias('superficie_media_ha'),
        F.max('superficie_ha').alias('superficie_max_ha'),
        F.sum(F.when(F.col('severidad_evento') == 'catastrofico', 1).otherwise(0)).alias('n_catastroficos'),
        F.avg('t2m_c').alias('t2m_medio'),
        F.avg('tp_mm').alias('tp_medio'),
        F.avg('t2m_anomaly_mensual').alias('t2m_anomaly_medio'),
        F.avg('tp_anomaly_mensual').alias('tp_anomaly_medio'),
        F.sum('es_temporada_incendios').alias('n_incendios_temporada_alta'),
    )
)

print(f'✓ {agg_comuna_anio.count()} combinaciones comuna-año')
agg_comuna_anio.orderBy(F.desc('superficie_total_ha')).show(10, truncate=False)


✓ 2617 combinaciones comuna-año
+----------+----------------+----+-----------+-------------------+-------------------+------------------+---------------+------------------+---------------------+---------------------+-------------------+--------------------------+
|region    |comuna          |año |n_incendios|superficie_total_ha|superficie_media_ha|superficie_max_ha |n_catastroficos|t2m_medio         |tp_medio             |t2m_anomaly_medio    |tp_anomaly_medio   |n_incendios_temporada_alta|
+----------+----------------+----+-----------+-------------------+-------------------+------------------+---------------+------------------+---------------------+---------------------+-------------------+--------------------------+
|Maule     |Cauquenes       |2017|35         |180372.8928        |5153.5112228571425 |159812.58         |4              |18.658245171428568|0.006905038497714287 |1.1141603340971427   |-0.9161185600000006|30                        |
|O'Higgins |Pumanque        |2017|3     

## 8. Funciones de ventana: score de estrés hídrico y ranking por año

- **`estres_hidrico_score`**: z-score de la anomalía de temperatura menos z-score de la
  anomalía de precipitación, calculado sobre toda la tabla comuna-año (`Window` sin
  partición). Valores altos = comuna-año más caliente y más seco de lo normal para esa
  celda climática — la señal de estrés hídrico que motiva el proyecto.
- **`ranking_severidad_anio`**: ranking de comunas dentro de cada año por superficie
  quemada total (`Window.partitionBy('año')`), para identificar las comunas más golpeadas
  temporada a temporada.


In [8]:
stats = agg_comuna_anio.select(
    F.avg('t2m_anomaly_medio').alias('t2m_mu'), F.stddev('t2m_anomaly_medio').alias('t2m_sd'),
    F.avg('tp_anomaly_medio').alias('tp_mu'), F.stddev('tp_anomaly_medio').alias('tp_sd'),
).collect()[0]

w_anio = Window.partitionBy('año').orderBy(F.desc('superficie_total_ha'))

agg_final = (
    agg_comuna_anio
    .withColumn('t2m_anomaly_z', (F.col('t2m_anomaly_medio') - F.lit(stats['t2m_mu'])) / F.lit(stats['t2m_sd']))
    .withColumn('tp_anomaly_z', (F.col('tp_anomaly_medio') - F.lit(stats['tp_mu'])) / F.lit(stats['tp_sd']))
    .withColumn('estres_hidrico_score', F.col('t2m_anomaly_z') - F.col('tp_anomaly_z'))
    .withColumn('ranking_severidad_anio', F.rank().over(w_anio))
)

print('Top 15 comuna-año por severidad (superficie_total_ha):')
agg_final.orderBy(F.desc('superficie_total_ha')).select(
    'año', 'region', 'comuna', 'n_incendios', 'superficie_total_ha', 'estres_hidrico_score'
).show(15, truncate=False)

print('Top 15 comuna-año por estrés hídrico (más caliente/seco de lo normal):')
agg_final.orderBy(F.desc('estres_hidrico_score')).select(
    'año', 'region', 'comuna', 'n_incendios', 'superficie_total_ha', 'estres_hidrico_score'
).show(15, truncate=False)


Top 15 comuna-año por severidad (superficie_total_ha):
+----+-------------+----------------+-----------+-------------------+--------------------+
|año |region       |comuna          |n_incendios|superficie_total_ha|estres_hidrico_score|
+----+-------------+----------------+-----------+-------------------+--------------------+
|2017|Maule        |Cauquenes       |35         |180372.8928        |1.2770706005409151  |
|2017|O'Higgins    |Pumanque        |3          |46490.350000000006 |1.4385774548465833  |
|2017|Maule        |Vichuquén       |26         |44400.829999999994 |2.0044356612753464  |
|2017|Biobío       |Florida         |29         |30514.382          |1.7989204751474988  |
|2011|Biobío       |Florida         |40         |24917.440000000002 |-1.1739522244687346 |
|2014|Maule        |Constitución    |66         |19241.528500000004 |-0.4125216951486552 |
|2017|Biobío       |Concepción      |18         |18958.164          |1.8189335098834112  |
|2011|Magallanes   |Torres del Pain

## 9. Guardar resultados

Dos salidas:
- Parquet particionado por `año` (formato eficiente para releer, mantiene tipos) — demuestra
  el patrón de escritura particionada de Spark.
- Un único CSV plano (`coalesce(1)`) para que el equipo de dashboard/BI lo abra directo en
  Tableau/PowerBI/pandas sin depender de Spark.


In [9]:
agg_final.write.mode('overwrite').partitionBy('año').parquet(f'{OUTPUT_DIR}/parquet')

(
    agg_final
    .coalesce(1)
    .write.mode('overwrite')
    .option('header', True)
    .csv(f'{OUTPUT_DIR}/csv')
)

print('✓ Guardado:')
print(f'  Parquet (particionado por año) -> {OUTPUT_DIR}/parquet')
print(f'  CSV (un solo archivo)           -> {OUTPUT_DIR}/csv')
print()
print(f'Filas finales: {agg_final.count()} combinaciones comuna-año')


✓ Guardado:
  Parquet (particionado por año) -> /content/drive/MyDrive/BigDataSIC/datos_procesados/spark_comuna_anio/parquet
  CSV (un solo archivo)           -> /content/drive/MyDrive/BigDataSIC/datos_procesados/spark_comuna_anio/csv

Filas finales: 2617 combinaciones comuna-año


## 10. Cerrar la sesión de Spark

In [ ]:
spark.stop()
print('Sesión de Spark cerrada.')


## ✅ Checklist de validación

- [x] Dataset de entrada cargado con el schema esperado (sección 5)
- [x] Filas sin comuna/región/año descartadas y contadas explícitamente (sección 6)
- [x] Agregación comuna-año sin perder el link a `región` (sección 7)
- [x] Score de estrés hídrico calculado con funciones de ventana reales de Spark, no en pandas (sección 8)
- [x] Salida escrita en dos formatos (parquet particionado + CSV plano) para el resto del equipo

**Próximo paso:** con `spark_comuna_anio/csv`, el equipo de dashboard puede graficar
`estres_hidrico_score` y `ranking_severidad_anio` por región/año, y el equipo de modelo puede
usar `agg_final` (o el parquet evento-a-evento del notebook 03) como base de features.

**Nota:** si alguien del equipo tiene el dataset histórico comuna-año 1984-2024 preparado
aparte (fuera de esta carpeta de Drive), se puede unir a `agg_final` por `comuna`+`año` para
dar contexto histórico más largo (los eventos CONAF de este cruce solo cubren 2010-2019) —
avísenme y agrego esa celda de join.
